In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/indra/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/indra/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/indra/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/indra/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number, count
from pyspark.sql.window import Window
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas5") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [5]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/indra/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df_transaksi.show(5)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



In [6]:
df_target = spark.createDataFrame(
    pd.DataFrame(data_target_cabang)
)

df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [7]:
# Menghitung total pendapatan setiap kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Menggabungkan dengan data target
hasil_target = ringkasan_kota.join(
    df_target,
    on="kota",
    how="inner"
)

# Menghitung persentase pencapaian target
hasil_target = hasil_target.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

# Mengurutkan dari pencapaian tertinggi
hasil_target.orderBy(
    col("pencapaian_persen").desc()
).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [8]:
# Menghitung total pendapatan berdasarkan kota dan kategori
pendapatan_kategori = df_transaksi.groupBy(
    "kota", "kategori"
).agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Membuat window berdasarkan kota
window_kota = Window.partitionBy("kota").orderBy(
    col("total_pendapatan").desc()
)

# Memberikan peringkat pada setiap kategori di masing-masing kota
kategori_terlaris = pendapatan_kategori.withColumn(
    "peringkat",
    row_number().over(window_kota)
)

# Mengambil kategori dengan peringkat pertama
kategori_terlaris.filter(
    col("peringkat") == 1
).orderBy("kota").show()

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



In [10]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'target'")

Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'target'


In [11]:
jumlah_transaksi = spark.sql('''
    SELECT
        t.kota,
        p.pic_cabang,
        COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target p
        ON t.kota = p.kota
    GROUP BY t.kota, p.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

jumlah_transaksi.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+

